In [1]:
import pickle
import matplotlib.pyplot as plt
from matplotlib.transforms import offset_copy
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe
import astropy.units as u
import numpy as np
import pandas as pd
plt.rcParams.update({'font.size': 10})

In [2]:
color_ice = '#D3D3D3'
color_ocean = '#20B2AA'
color_mantle = '#FFA07A'
color_core = '#F08080'
color_core2 = '#EC5F5F'
color_core3 = '#E84141'
color_sulfur = '#DEB841' #'#A4831C'

larger_font = 12
smaller_font = 8

include_B = False

In [3]:
with open(f'Triton_models/model_nominal_FeSLayer.pkl', 'rb') as f:
    nominal_FeSLayer = pickle.load(f)

In [4]:
fig_R_dims = (10, 5)

if include_B:
    fig_R, (ax_B, ax_R) = plt.subplots(nrows=2, ncols=1, height_ratios=[1, 4], sharex=True, figsize=fig_R_dims, dpi=300)
else:
    fig_R, ax_R = plt.subplots(figsize=fig_R_dims, dpi=300)

ax_R.axhspan(nominal_FeSLayer.pb.R_O.to(u.km).value, nominal_FeSLayer.pb.R_I.to(u.km).value, color=color_ice, zorder=1)
ax_R.axhspan(nominal_FeSLayer.pb.R_M.to(u.km).value, nominal_FeSLayer.pb.R_O.to(u.km).value, color=color_ocean, zorder=1)
ax_R.axhspan(nominal_FeSLayer.pb.R_C.to(u.km).value, nominal_FeSLayer.pb.R_M.to(u.km).value, color=color_mantle, zorder=1)
ax_R.axhspan(0, nominal_FeSLayer.pb.R_C.to(u.km).value, color=color_core, zorder=1)

t_arr = nominal_FeSLayer.pb.t_arr.to(u.Gyr)

FeS_layer_x_index = np.argmax(nominal_FeSLayer.pb.R_IC_arr.to(u.km).value < nominal_FeSLayer.pb.R_C.to(u.km).value)
FeS_layer_x = (t_arr.value[FeS_layer_x_index] + 9) / 2
FeS_layer_y_index = int(np.argmin(np.abs(t_arr.value - FeS_layer_x)))
FeS_layer_y = (nominal_FeSLayer.pb.R_IC_arr.to(u.km).value[FeS_layer_y_index] + nominal_FeSLayer.pb.R_C.to(u.km).value) / 2

ax_R.fill_between(t_arr.value[FeS_layer_x_index:], nominal_FeSLayer.pb.R_IC_arr.to(u.km).value[FeS_layer_x_index:], nominal_FeSLayer.pb.R_C.to(u.km).value, facecolor=color_core2, edgecolor='face', zorder=2)
ax_R.text(FeS_layer_x, FeS_layer_y, 'FeS Layer', color='black', ha='center', va='center_baseline', zorder=3)

X, Y = np.meshgrid(t_arr, nominal_FeSLayer.pb.R_arr.to(u.km)[1:])
Z = np.array(np.array(nominal_FeSLayer.pb.T_mat[1:, :]).T) * u.K
T_contours = np.arange(500, 2500, 250) * u.K

ax_R.axvline(4.5, color='k', linestyle='dotted', linewidth=1, zorder=3)
ax_R.text(4.5, nominal_FeSLayer.pb.R_I.to(u.km).value, 'Present', ha='left', va='top', transform=offset_copy(ax_R.transData, fig=fig_R, x=2, y=-2, units='points'), zorder=3)

CS = ax_R.contour(X.value, Y.value, Z.T.value, T_contours.value, colors='black', linewidths=1, linestyles='dashed', zorder=3)
ax_R.clabel(CS, levels=CS.levels[::2], inline=True, fontsize=smaller_font, fmt=lambda val: f'{int(val)} K')
ax_R.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_R.set_ylabel('Radial Distance (km)', fontsize=larger_font)
ax_R.set_xlim([0, 9])
ax_R.set_ylim([0, nominal_FeSLayer.pb.R_I.to(u.km).value])

if include_B:
    # ax_B = ax_R.twinx()
    ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
    ax_B.plot(t_arr.value, nominal_FeSLayer.pb.B_S_arr.to(u.uT).value, color='black', linewidth=2)
    ax_B.set_ylabel('B$_S$ ($\mathregular{\mu}$T)', fontsize=larger_font)
    ax_B.set_ylim([0, 4])
    #ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
    #ax_B.tick_params(axis='y')
    
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeSLayer_B.pdf')
    
else:
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeSLayer.pdf')

In [5]:
fig_combined_dims = (10, 5)
fig_combined, ((ax_Q, ax_T), (ax_Rem, ax_B)) = plt.subplots(2, 2, figsize=fig_combined_dims, sharex='all', dpi=300)


t_arr = nominal_FeSLayer.pb.t_arr.to(u.Gyr)

ax_Q.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Q.plot(t_arr.value, nominal_FeSLayer.pb.Q_C_arr.to(u.GW).value, color=color_core, label='Core-Mantle Boundary', linewidth=2.5)
ax_Q.plot(t_arr.value, nominal_FeSLayer.pb.Q_ad_arr.to(u.GW).value, color='k', label='Adiabatic Heat Flow in the Core', linewidth=2.5)
ax_Q.axhline(0, color='black', linewidth=1, linestyle='dashed')
ax_Q.set_ylabel('Heat Flow (GW)', fontsize=larger_font)
ax_Q.set_ylim(-15, 15)
ax_Q.tick_params(axis='y')
ax_Q.legend(loc='lower right')

T_cmb = nominal_FeSLayer.pb.T_mat[0, :]
T_mid = nominal_FeSLayer.pb.T_mat[round(len(nominal_FeSLayer.pb.R_arr) / 2), :]
ax_T.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_T.plot(t_arr.value, T_cmb.value, label='Core-Mantle Boundary', color=color_core, linewidth=2.5)
# ax_T.plot(t_arr.value, nominal_FeS.pb.T_mat[5, :].value, label='CMB + 30 km', color=color_mantle, linestyle='dashed')
ax_T.plot(t_arr.value, T_mid.value, label='Mid-Mantle', color=color_mantle, linewidth=2.5)
ax_T.axhline(nominal_FeSLayer.pb.T_liq_CMB.value, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_T.text(9.0, nominal_FeSLayer.pb.T_liq_CMB.value, 'Liquidus Temperature', ha='right', va='bottom', transform=offset_copy(ax_T.transData, fig=fig_combined, x=-2, y=2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_T.set_ylabel('Temperature (K)', fontsize=larger_font)
ax_T.set_ylim([750, 2250])
ax_T.set_yticks([750, 1000, round(nominal_FeSLayer.pb.T_liq_CMB.value), 1500, 2000, 2250], labels=[750, 1000, round(nominal_FeSLayer.pb.T_liq_CMB.value), 1500, 2000, 2250])
ax_T.tick_params(axis='y')
ax_T.legend(loc='upper right')

ax_Rem.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Rem.plot(t_arr.value, nominal_FeSLayer.pb.Re_m_arr.value, color=color_ocean, linewidth=2.5)
ax_Rem.axhline(40, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_Rem.axhline(100, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(150, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(200, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.text(9.0, 40, 'Critical $\mathregular{Re_M}$', ha='right', va='top', transform=offset_copy(ax_Rem.transData, fig=fig_combined, x=-2, y=-2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_Rem.set_ylabel('Magnetic\nReynolds Number', fontsize=larger_font)
ax_Rem.set_ylim(0, 400)
ax_Rem.set_yticks([0, 40, 100, 150, 200, 300, 400], labels=[0, 40, 100, 150, 200, 300, 400])
ax_Rem.tick_params(axis='y')

ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_B.plot(t_arr.value, nominal_FeSLayer.pb.B_S_arr.to(u.uT).value, color=color_ocean, linewidth=2.5)
ax_B.set_ylabel('Surface Magnetic\nField Strength ($\mathregular{\mu}$T)', fontsize=larger_font)
ax_B.set_ylim(0, 4)
ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
ax_B.tick_params(axis='y')

ax_Rem.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_B.set_xlabel('Time (Gyr)', fontsize=larger_font)

ax_Rem.set_xlim([0, 9])
ax_Rem.set_xticks(np.arange(0, 10, 1))

fig_combined.tight_layout()
fig_combined.savefig('Results2_FeSLayer.pdf')

In [6]:
with open(f'Triton_models/model_nominal_FeSnow.pkl', 'rb') as f:
    nominal_FeSnow = pickle.load(f)

In [7]:
fig_R_dims = (10, 5)

if include_B:
    fig_R, (ax_B, ax_R) = plt.subplots(nrows=2, ncols=1, height_ratios=[1, 4], sharex=True, figsize=fig_R_dims, dpi=300)
else:
    fig_R, ax_R = plt.subplots(figsize=fig_R_dims, dpi=300)

ax_R.axhspan(nominal_FeSnow.pb.R_O.to(u.km).value, nominal_FeSnow.pb.R_I.to(u.km).value, color=color_ice, zorder=1)
ax_R.axhspan(nominal_FeSnow.pb.R_M.to(u.km).value, nominal_FeSnow.pb.R_O.to(u.km).value, color=color_ocean, zorder=1)
ax_R.axhspan(nominal_FeSnow.pb.R_C.to(u.km).value, nominal_FeSnow.pb.R_M.to(u.km).value, color=color_mantle, zorder=1)
ax_R.axhspan(0, nominal_FeSnow.pb.R_C.to(u.km).value, color=color_core, zorder=1)

t_arr = nominal_FeSnow.pb.t_arr.to(u.Gyr)

ax_R.fill_between(t_arr.value[nominal_FeSnow.pb.T_liq_mask], nominal_FeSnow.pb.R_C.to(u.km).value - nominal_FeSnow.pb.HSt_arr.to(u.km).value, nominal_FeSnow.pb.R_C.to(u.km).value, facecolor=color_core2, zorder=2, edgecolor='face')

stable_x = (min(t_arr.value[nominal_FeSnow.pb.T_liq_mask]) + 9) / 2
ax_R.text(stable_x, nominal_FeSnow.pb.R_C.to(u.km).value / 2, 'Fe Snow', color='black', ha='center', va='center_baseline', zorder=3)

X, Y = np.meshgrid(t_arr, nominal_FeSnow.pb.R_arr.to(u.km)[1:])
Z = np.array(np.array(nominal_FeSnow.pb.T_mat[1:, :]).T) * u.K
T_contours = np.arange(500, 2500, 250) * u.K

ax_R.axvline(4.5, color='k', linestyle='dotted', linewidth=1, zorder=3)
ax_R.text(4.5, nominal_FeSnow.pb.R_I.to(u.km).value, 'Present', ha='left', va='top', transform=offset_copy(ax_R.transData, fig=fig_R, x=2, y=-2, units='points'), zorder=3)

CS = ax_R.contour(X.value, Y.value, Z.T.value, T_contours.value, colors='black', linewidths=1, linestyles='dashed', zorder=3)
ax_R.clabel(CS, levels=CS.levels[::2], inline=True, fontsize=smaller_font, fmt=lambda val: f'{int(val)} K')
ax_R.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_R.set_ylabel('Radial Distance (km)', fontsize=larger_font)
ax_R.set_xlim([0, 9])
ax_R.set_ylim([0, nominal_FeSnow.pb.R_I.to(u.km).value])

if include_B:
    # ax_B = ax_R.twinx()
    ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
    ax_B.plot(t_arr.value, nominal_FeSnow.pb.B_S_arr.to(u.uT).value, color='black', linewidth=2)
    ax_B.set_ylabel('B$_S$ ($\mathregular{\mu}$T)', fontsize=larger_font)
    ax_B.set_ylim([0, 4])
    #ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
    #ax_B.tick_params(axis='y')
    
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeSnow_B.pdf')
    
else:
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeSnow.pdf')

In [8]:
fig_combined_dims = (10, 5)
fig_combined, ((ax_Q, ax_T), (ax_Rem, ax_B)) = plt.subplots(2, 2, figsize=fig_combined_dims, sharex='all', dpi=300)

t_arr = nominal_FeSnow.pb.t_arr.to(u.Gyr)

ax_Q.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Q.plot(t_arr.value, nominal_FeSnow.pb.Q_C_arr.to(u.GW).value, color=color_core, label='Core-Mantle Boundary', linewidth=2.5)
ax_Q.plot(t_arr.value, nominal_FeSnow.pb.Q_ad_arr.to(u.GW).value, color='k', label='Adiabatic Heat Flow in the Core', linewidth=2.5)
ax_Q.axhline(0, color='black', linewidth=1, linestyle='dashed')
ax_Q.set_ylabel('Heat Flow (GW)', fontsize=larger_font)
ax_Q.set_ylim(-15, 15)
ax_Q.tick_params(axis='y')
ax_Q.legend(loc='lower right')

T_cmb = nominal_FeSnow.pb.T_mat[0, :]
T_mid = nominal_FeSnow.pb.T_mat[round(len(nominal_FeSnow.pb.R_arr) / 2), :]
ax_T.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_T.plot(t_arr.value, T_cmb.value, label='Core-Mantle Boundary', color=color_core, linewidth=2.5)
# ax_T.plot(t_arr.value, nominal_FeS.pb.T_mat[5, :].value, label='CMB + 30 km', color=color_mantle, linestyle='dashed')
ax_T.plot(t_arr.value, T_mid.value, label='Mid-Mantle', color=color_mantle, linewidth=2.5)
ax_T.axhline(nominal_FeSnow.pb.T_liq_CMB.value, nominal_FeSnow.t_start.to(u.Gyr).value, nominal_FeSnow.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_T.text(9.0, nominal_FeSnow.pb.T_liq_CMB.value, 'Liquidus Temperature', ha='right', va='bottom', transform=offset_copy(ax_T.transData, fig=fig_combined, x=-2, y=2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_T.set_ylabel('Temperature (K)', fontsize=larger_font)
ax_T.set_ylim([750, 2250])
ax_T.set_yticks([750, 1000, round(nominal_FeSnow.pb.T_liq_CMB.value), 1500, 2000, 2250], labels=[750, 1000, round(nominal_FeSnow.pb.T_liq_CMB.value), 1500, 2000, 2250])
ax_T.tick_params(axis='y')
ax_T.legend(loc='upper right')

ax_Rem.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Rem.plot(t_arr.value, nominal_FeSnow.pb.Re_m_arr.value, color=color_ocean, linewidth=2.5)
ax_Rem.axhline(40, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_Rem.axhline(100, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(150, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(200, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.text(9.0, 40, 'Critical $\mathregular{Re_M}$', ha='right', va='top', transform=offset_copy(ax_Rem.transData, fig=fig_combined, x=-2, y=-2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_Rem.set_ylabel('Magnetic\nReynolds Number', fontsize=larger_font)
ax_Rem.set_ylim(0, 400)
ax_Rem.set_yticks([0, 40, 100, 150, 200, 300, 400], labels=[0, 40, 100, 150, 200, 300, 400])
ax_Rem.tick_params(axis='y')

ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_B.plot(t_arr.value, nominal_FeSnow.pb.B_S_arr.to(u.uT).value, color=color_ocean, linewidth=2.5)
ax_B.set_ylabel('Surface Magnetic\nField Strength ($\mathregular{\mu}$T)', fontsize=larger_font)
ax_B.set_ylim(0, 4)
ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
ax_B.tick_params(axis='y')

ax_Rem.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_B.set_xlabel('Time (Gyr)', fontsize=larger_font)

ax_Rem.set_xlim([0, 9])
ax_Rem.set_xticks(np.arange(0, 10, 1))

fig_combined.tight_layout()
fig_combined.savefig('Results2_FeSnow.pdf')

In [9]:
with open(f'Triton_models/model_nominal_FeInner.pkl', 'rb') as f:
    nominal_FeInner = pickle.load(f)

In [10]:
fig_R_dims = (10, 5)

if include_B:
    fig_R, (ax_B, ax_R) = plt.subplots(nrows=2, ncols=1, height_ratios=[1, 4], sharex=True, figsize=fig_R_dims, dpi=300)
else:
    fig_R, ax_R = plt.subplots(figsize=fig_R_dims, dpi=300)

ax_R.axhspan(nominal_FeInner.pb.R_O.to(u.km).value, nominal_FeInner.pb.R_I.to(u.km).value, color=color_ice, zorder=1)
ax_R.axhspan(nominal_FeInner.pb.R_M.to(u.km).value, nominal_FeInner.pb.R_O.to(u.km).value, color=color_ocean, zorder=1)
ax_R.axhspan(nominal_FeInner.pb.R_C.to(u.km).value, nominal_FeInner.pb.R_M.to(u.km).value, color=color_mantle, zorder=1)
ax_R.axhspan(0, nominal_FeInner.pb.R_C.to(u.km).value, color=color_core, zorder=1)

t_arr = nominal_FeInner.pb.t_arr.to(u.Gyr)

inner_core_x_index = np.argmax(nominal_FeInner.pb.R_IC_arr.to(u.km).value > 0)
inner_core_x = (t_arr.value[inner_core_x_index] + 9) / 2
inner_core_y_index = int((inner_core_x_index + len(nominal_FeInner.pb.R_IC_arr)) / 2)
inner_core_y = nominal_FeInner.pb.R_IC_arr.to(u.km).value[inner_core_y_index] / 2

ax_R.fill_between(t_arr.value[inner_core_x_index:], 0, nominal_FeInner.pb.R_IC_arr.to(u.km).value[inner_core_x_index:], facecolor=color_core2, edgecolor='face', zorder=2)

ax_R.text(inner_core_x, inner_core_y, 'Fe Inner Core', color='black', ha='center', va='center_baseline', zorder=3)

X, Y = np.meshgrid(t_arr, nominal_FeInner.pb.R_arr.to(u.km)[1:])
Z = np.array(np.array(nominal_FeInner.pb.T_mat[1:, :]).T) * u.K
T_contours = np.arange(500, 2500, 250) * u.K

ax_R.axvline(4.5, color='k', linestyle='dotted', linewidth=1, zorder=3)
ax_R.text(4.5, nominal_FeInner.pb.R_I.to(u.km).value, 'Present', ha='left', va='top', transform=offset_copy(ax_R.transData, fig=fig_R, x=2, y=-2, units='points'), zorder=3)

CS = ax_R.contour(X.value, Y.value, Z.T.value, T_contours.value, colors='black', linewidths=1, linestyles='dashed', zorder=3)
ax_R.clabel(CS, levels=CS.levels[::2], inline=True, fontsize=smaller_font, fmt=lambda val: f'{int(val)} K')
ax_R.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_R.set_ylabel('Radial Distance (km)', fontsize=larger_font)
ax_R.set_xlim([0, 9])
ax_R.set_ylim([0, nominal_FeInner.pb.R_I.to(u.km).value])
    
if include_B:
    # ax_B = ax_R.twinx()
    ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
    ax_B.plot(t_arr.value, nominal_FeInner.pb.B_S_arr.to(u.uT).value, color='black', linewidth=2)
    ax_B.set_ylabel('B$_S$ ($\mathregular{\mu}$T)', fontsize=larger_font)
    ax_B.set_ylim([0, 4])
    #ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
    #ax_B.tick_params(axis='y')
    
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeInner_B.pdf')
    
else:
    fig_R.tight_layout()
    fig_R.savefig('Results1_FeInner.pdf')

In [11]:
fig_combined_dims = (10, 5)
fig_combined, ((ax_Q, ax_T), (ax_Rem, ax_B)) = plt.subplots(2, 2, figsize=fig_combined_dims, sharex='all', dpi=300)

t_arr = nominal_FeInner.pb.t_arr.to(u.Gyr)

ax_Q.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Q.plot(t_arr.value, nominal_FeInner.pb.Q_C_arr.to(u.GW).value, color=color_core, label='Core-Mantle Boundary', linewidth=2.5)
ax_Q.plot(t_arr.value, nominal_FeInner.pb.Q_ad_arr.to(u.GW).value, color='k', label='Adiabatic Heat Flow in the Core', linewidth=2.5)
ax_Q.axhline(0, color='black', linewidth=1, linestyle='dashed')
ax_Q.set_ylabel('Heat Flow (GW)', fontsize=larger_font)
ax_Q.set_ylim(-15, 15)
ax_Q.tick_params(axis='y')
ax_Q.legend(loc='lower right')

T_cmb = nominal_FeInner.pb.T_mat[0, :]
T_mid = nominal_FeInner.pb.T_mat[round(len(nominal_FeInner.pb.R_arr) / 2), :]
ax_T.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_T.plot(t_arr.value, T_cmb.value, label='Core-Mantle Boundary', color=color_core, linewidth=2.5)
# ax_T.plot(t_arr.value, nominal_FeS.pb.T_mat[5, :].value, label='CMB + 30 km', color=color_mantle, linestyle='dashed')
ax_T.plot(t_arr.value, T_mid.value, label='Mid-Mantle', color=color_mantle, linewidth=2.5)
ax_T.axhline(nominal_FeInner.pb.T_liq_CMB.value, nominal_FeInner.t_start.to(u.Gyr).value, nominal_FeInner.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_T.text(9.0, nominal_FeInner.pb.T_liq_CMB.value, 'Liquidus Temperature', ha='right', va='bottom', transform=offset_copy(ax_T.transData, fig=fig_combined, x=-2, y=2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_T.set_ylabel('Temperature (K)', fontsize=larger_font)
ax_T.set_ylim([750, 2250])
ax_T.set_yticks([750, 1000, round(nominal_FeInner.pb.T_liq_CMB.value), 1500, 2000, 2250], labels=[750, 1000, round(nominal_FeInner.pb.T_liq_CMB.value), 1500, 2000, 2250])
ax_T.tick_params(axis='y')
ax_T.legend(loc='upper right')

ax_Rem.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_Rem.plot(t_arr.value, nominal_FeInner.pb.Re_m_arr.value, color=color_ocean, linewidth=2.5)
ax_Rem.axhline(40, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed')
ax_Rem.axhline(100, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(150, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.axhline(200, nominal_FeSLayer.t_start.to(u.Gyr).value, nominal_FeSLayer.t_end.to(u.Gyr).value, color='black', linewidth=1, linestyle='dashed', alpha=0.5)
ax_Rem.text(9.0, 40, 'Critical $\mathregular{Re_M}$', ha='right', va='top', transform=offset_copy(ax_Rem.transData, fig=fig_combined, x=-2, y=-2, units='points'), path_effects=[pe.withStroke(linewidth=2, foreground='white')])
ax_Rem.set_ylabel('Magnetic\nReynolds Number', fontsize=larger_font)
ax_Rem.set_ylim(0, 400)
ax_Rem.set_yticks([0, 40, 100, 150, 200, 300, 400], labels=[0, 40, 100, 150, 200, 300, 400])
ax_Rem.tick_params(axis='y')

ax_B.axvline(4.5, color='black', linewidth=1, linestyle='dotted')
ax_B.plot(t_arr.value, nominal_FeInner.pb.B_S_arr.to(u.uT).value, color=color_ocean, linewidth=2.5)
ax_B.set_ylabel('Surface Magnetic\nField Strength ($\mathregular{\mu}$T)', fontsize=larger_font)
ax_B.set_ylim(0, 4)
ax_B.set_yticks(sorted(set(range(0, 5, 1))), labels=sorted(set(range(0, 5, 1))))
ax_B.tick_params(axis='y')

ax_Rem.set_xlabel('Time (Gyr)', fontsize=larger_font)
ax_B.set_xlabel('Time (Gyr)', fontsize=larger_font)

ax_Rem.set_xlim([0, 9])
ax_Rem.set_xticks(np.arange(0, 10, 1))

fig_combined.tight_layout()
fig_combined.savefig('Results2_FeInner.pdf')

In [68]:
Rem_C = 200
all_df = pd.read_csv(f'Triton_results_RemC{Rem_C}.csv')

FeInner_df = all_df[all_df['core'] == 'FeInner']
FeSnow_df = all_df[all_df['core'] == 'FeSnow']
FeSLayer_df = all_df[all_df['core'] == 'FeSLayer']

In [69]:
var_list = ['T_Ci', 'T_Mi', 'rad_coeff', 'wtS_C', 'k_C']

In [70]:
FeInner_plot = {}
FeSnow_plot = {}
FeSLayer_plot = {}

plot_dicts = [(FeInner_plot, FeInner_df), (FeSnow_plot, FeSnow_df), (FeSLayer_plot, FeSLayer_df)]

for var in var_list:
    for plot_dict, core_df in plot_dicts:
        list_a = list(core_df[(core_df['var'] == var) | (core_df['var'] == 'nominal')][var])
        list_b = list(core_df[(core_df['var'] == var) | (core_df['var'] == 'nominal')]['dynamo_cessation_time'])
        list_ab = sorted(zip(list_a, list_b))
        plot_dict[var] = list(zip(*list_ab))
        
        nom_index = 0 if var == 'k_C' else len(list_a) // 2
        var_nom = plot_dict[var][0][nom_index]
        var_min = plot_dict[var][0][0]
        var_max = plot_dict[var][0][-1]
        
        var_min_abs = abs(var_nom - var_min)
        var_max_abs = abs(var_nom - var_max)
        
        var_abs = var_min_abs if var_min_abs > var_max_abs else var_max_abs
        var_min = var_nom - var_abs
        var_max = var_nom + var_abs
        
        if var_min != var_max:
            plot_dict[var].append(tuple([(v - var_min) / (var_max - var_min) for v in plot_dict[var][0]]))
        else:
            plot_dict[var].append(tuple([0.5]))
            

In [71]:
plot_format_dict = {'T_Ci': [f'Initial Core Temperature (1800 to 2600 K)', color_core, 6],
                    'T_Mi': [f'Initial Mantle Temperature (1600 to 2000 K)', color_mantle, 5],
                    'rad_coeff': [f'Radioactivity Coefficient (0 to 2 × nominal)', color_ocean, 4],
                    'wtS_C': [f'Core Sulfur Content (S-Rich: 30 to 36 wt%, S-Poor: 0 to 25 wt%)', color_sulfur, 3],
                    'k_C': [f'Core Thermal Conductivity (5, 15, 30 W/m/K)', color_ice, 2]}

In [72]:
fig_sens_dims = (10, 5)
fig_sens, ax_sens = plt.subplots(figsize=fig_sens_dims, dpi=300)

ax_sens.axhline(4.5, color='black', linewidth=1, linestyle='dotted', alpha=0.5)
ax_sens.set_aspect('auto')

for k, v in FeSLayer_plot.items():
    ax_sens.plot(v[2], v[1], marker='o', markersize=7.5, lw=1, color=plot_format_dict[k][1], zorder=plot_format_dict[k][2])

for k, v in FeSnow_plot.items():
    ax_sens.plot(v[2], v[1], marker='^', markersize=7.5, lw=1, color=plot_format_dict[k][1], zorder=plot_format_dict[k][2])

for k, v in FeInner_plot.items():
    ax_sens.plot(v[2], v[1], marker='s', markersize=7.5, lw=1, color=plot_format_dict[k][1], zorder=plot_format_dict[k][2])

legend_handles = []
for k, v in plot_format_dict.items():
    legend_handles.append(Line2D([0], [0], marker='o', markersize=7.5, lw=1, color=plot_format_dict[k][1], label=plot_format_dict[k][0]))

ax_sens.text(0.5, FeSLayer_plot['rad_coeff'][1][2] + 0.5, 'FeS Layer (●)', color='black', va='center_baseline', ha='center', zorder=7)
ax_sens.text(0.5, FeSnow_plot['rad_coeff'][1][2] - 0.5, 'Fe Snow (▲)', color='black', va='center_baseline', ha='center', zorder=7)
ax_sens.text(0.5, FeInner_plot['rad_coeff'][1][2] + 0.5, 'Fe Inner Core (■)', color='black', va='center_baseline', ha='center', zorder=7)

ax_sens.set_ylim([0, 14])
ax_sens.legend(loc='upper right', handles=legend_handles, ncol=2, fontsize=smaller_font)
ax_sens.set_xlabel('Parameter Value (Normalized)')
ax_sens.set_ylabel('Model Time of Dynamo Cessation (Gyr)')

fig_sens.tight_layout()
fig_sens.savefig(f'Sensitivity_RemC{Rem_C}.pdf')